In [1]:
# Install required build tools and Boost libraries
!apt-get update -qq
!apt-get install -y swig libboost-all-dev

# Re-install vina
!pip install vina

E: List directory /var/lib/apt/lists/partial is missing. - Acquire (13: Permission denied)
E: Could not open lock file /var/lib/dpkg/lock-frontend - open (13: Permission denied)
E: Unable to acquire the dpkg frontend lock (/var/lib/dpkg/lock-frontend), are you root?
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 37.3 MB/s eta 0:00:0000:0100:01


In [2]:
# 1. Install OpenBabel Python bindings directly via pip (No conda/sudo required)
!pip install -q openbabel vina pandas

# 2. Convert SDF library to individual PDBQT files
import os
from openbabel import pybel

os.makedirs("sancdb_pdbqt", exist_ok=True)

count = 0
for i, mol in enumerate(pybel.readfile("sdf", "sancdb_prepared_3d.sdf")):
    mol.write("pdbqt", f"sancdb_pdbqt/sanc_{i+1}.pdbqt", overwrite=True)
    count += 1

print(f"Successfully converted {count} ligands into 'sancdb_pdbqt/' directory!")

Successfully converted 1002 ligands into 'sancdb_pdbqt/' directory!


In [4]:
import glob
files = glob.glob("sancdb_pdbqt/*.pdbqt")
print(f"Number of PDBQT ligand files found: {len(files)}")

Number of PDBQT ligand files found: 1002


In [1]:
import os
import glob
import pandas as pd
from openbabel import pybel
from vina import Vina
from concurrent.futures import ProcessPoolExecutor, as_completed

# --- CONFIGURATION ---
SDF_FILE = "sancdb_prepared_3d.sdf"
# Auto-detect receptor filename casing (1HCK_receptor.pdbqt or 1hck_receptor.pdbqt)
RECEPTOR_FILE = "1HCK_receptor.pdbqt" if os.path.exists("1HCK_receptor.pdbqt") else "1hck_receptor.pdbqt"
LIGAND_DIR = "sancdb_pdbqt"
OUTPUT_DIR = "docked_poses"
CSV_OUTPUT = "sancdb_1hck_ranked_scores.csv"

CENTER = [-9.344334, -10.6396675, 8.487167]
GRID_SIZE = [20.0, 20.0, 20.0]

os.makedirs(LIGAND_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. FAST CONVERT & SANITIZE (Strips invalid COMPND tags on the fly)
print("Converting & sanitizing SDF ligands...")
count = 0
for i, mol in enumerate(pybel.readfile("sdf", SDF_FILE)):
    pdbqt_str = mol.write("pdbqt")
    # Remove lines starting with COMPND that crash Vina
    clean_lines = [line for line in pdbqt_str.splitlines(keepends=True) if not line.startswith("COMPND")]
    
    out_path = os.path.join(LIGAND_DIR, f"sanc_{i+1}.pdbqt")
    with open(out_path, "w") as f:
        f.writelines(clean_lines)
    count += 1
print(f"Prepared {count} sanitized ligand files in '{LIGAND_DIR}/'.")

# 2. WORKER FUNCTION FOR MULTI-CORE DOCKING
def dock_single_ligand(ligand_path):
    ligand_id = os.path.splitext(os.path.basename(ligand_path))[0]
    out_pose_path = os.path.join(OUTPUT_DIR, f"{ligand_id}_docked.pdbqt")
    try:
        v = Vina(sf_name='vina')
        v.set_receptor(RECEPTOR_FILE)
        v.compute_vina_maps(center=CENTER, box_size=GRID_SIZE)
        v.set_ligand_from_file(ligand_path)
        v.dock(exhaustiveness=8, n_poses=1)
        v.write_poses(out_pose_path, n_poses=1, overwrite=True)
        best_affinity = v.energies()[0][0]
        return {"Ligand_ID": ligand_id, "Binding_Affinity_kcal_mol": best_affinity}
    except Exception:
        return None

# 3. PARALLEL EXECUTION ACROSS ALL CPU CORES
ligand_files = glob.glob(os.path.join(LIGAND_DIR, "*.pdbqt"))
num_cores = os.cpu_count() or 4
print(f"Launching parallel screening on {num_cores} CPU threads...")

results = []
with ProcessPoolExecutor(max_workers=num_cores) as executor:
    futures = {executor.submit(dock_single_ligand, f): f for f in ligand_files}
    completed = 0
    for future in as_completed(futures):
        res = future.result()
        if res:
            results.append(res)
        completed += 1
        if completed % 100 == 0 or completed == len(ligand_files):
            print(f"Progress: {completed}/{len(ligand_files)} compounds docked...")

# 4. EXPORT & RANK RESULTS
df = pd.DataFrame(results).sort_values(by="Binding_Affinity_kcal_mol", ascending=True)
df.to_csv(CSV_OUTPUT, index=False)

print(f"\nScreening complete! Processed {len(df)} compounds.")
print(f"Ranked results saved to: {CSV_OUTPUT}\n")
print("Top 10 Ranked Hits:")
print(df.head(10))

Converting & sanitizing SDF ligands...
Prepared 1002 sanitized ligand files in 'sancdb_pdbqt/'.
Launching parallel screening on 32 CPU threads...
Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... done.
done.
done.
done.
done.
done.
done.
done.
done.
done.
done.
done.
done.
done.
do

Performing docking (random seed: 1467557525) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
*Computing Vina grid ... Computing Vina grid ... done.
Computing Vina grid ... done.
*done.
done.
*Computing Vina grid ... done.
Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... *Computing Vina grid ... done.
Computing Vina grid ... *****Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... **Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... *Computing Vina grid ... *Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... *Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... *Computing Vina grid ... **Computing Vina grid ... *Computing Vina grid ... Computing Vina grid ... *****Computing Vina grid ... ***********Performing docking (random seed: -608137156) ... 
0% 

**************
done.
done.

mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -8.849          0          0
Computing Vina grid ... done.
Computing Vina grid ... done.
done.
done.
done.
done.
done.
*done.
done.
Computing Vina grid ... done.
done.
done.
done.
Computing Vina grid ... Computing Vina grid ... done.
Computing Vina grid ... done.
Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... done.
Computing Vina grid ... done.
done.
done.
done.
Computing Vina grid ... done.
done.
done.
Computing Vina grid ... Computing Vina grid ... done.
Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... done.
done.
Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Performing docking (random seed: -84167134) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
*done.
*Computing Vina grid ... *Computin

*Computing Vina grid ... ****Computing Vina grid ... Computing Vina grid ... **Computing Vina grid ... ****Computing Vina grid ... Computing Vina grid ... *************done.
*******Performing docking (random seed: -1307244775) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***

**********done.
********done.
**done.
*done.
done.
done.
done.
*Computing Vina grid ... *done.
*done.
*done.
done.
done.
Computing Vina grid ... *done.
done.
*
done.
*done.
Computing Vina grid ... *Performing docking (random seed: -618007481) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
*Computing Vina grid ... done.
*done.
**done.
done.
Computing Vina grid ... *Computing Vina grid ... *

...

***********Computing Vina grid ... **done.
******Computing Vina grid ... *****done.
Computing Vina grid ... ********done.
*****done.
****done.
*Computing Vina grid ... *done.
****
***done.
*Computing Vina grid ... **done.
*Computing Vina grid ... Computing Vina grid ... *done.
**Computing Vina grid ... done.
*Computing Vina grid ... 
mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -4.627          0          0
Computing Vina grid ... *****Computing Vina grid ... Computing Vina grid ... *done.
**Computing Vina grid ... **
**Computing Vina grid ... ******done.
*done.
*Computing Vina grid ... **
mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -8.837          0          0
Computing Vina grid ... **done.
*Computing Vina grid ... *Computing Vina grid ... done.
Computing Vina grid ... *
done.
Performing docking (random seed: -

******done.
*******
mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -9.466          0          0
Computing Vina grid ... **done.
Computing Vina grid ... ***done.
**
****done.
*********Computing Vina grid ... *Computing Vina grid ... done.
***Performing docking (random seed: -190902104) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
*

****
mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -7.096          0          0
Computing Vina grid ... *****done.
***done.
*done.
*Computing Vina grid ... *done.
***
**Computing Vina grid ... done.
done.
Performing docking (random seed: 1884276675) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
done.


**Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... done.
*
mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -8.493          0          0
Computing Vina grid ... done.
done.
done.
Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... *done.
*Computing Vina grid ... *done.
Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... **done.
Computing Vina grid ... **done.
done.
Computing Vina grid ... *done.
**done.
*Computing Vina grid ... Computing Vina grid ... done.
Computing Vina grid ... Computing Vina grid ... ***done.
done.
*Computing Vina grid ... **done.
done.
*Computing Vina grid ... done.
Computing Vina grid ... done.
Computing Vina grid ... *done.
*done.
*Computing Vina grid ... *done.
*done.
done.
Computing Vina grid ... done.
Computing Vina grid ... Computing Vina grid ... done.
done.
*Computing Vina grid ... done.


*done.
****done.
**done.
*done.
**done.
done.
***done.
******done.
**done.
done.
******done.
**done.
*done.
done.
********************************************************************************
************************************************************************************************************
*
***********
*******************************************************
***************
************************************
****************
***************
Progress: 1000/1002 compounds docked...
**********************************
*******************
Progress: 1002/1002 compounds docked...

Screening complete! Processed 52 compounds.
Ranked results saved to: sancdb_1hck_ranked_scores.csv

Top 10 Ranked Hits:
   Ligand_ID  Binding_Affinity_kcal_mol
22  sanc_118                    -10.821
13   sanc_40                    -10.108
1   sanc_704                    -10.091
3   sanc_604                    -10.043
17  sanc_447                     -9.893
18   sanc_41                     -9.599
2   

In [3]:
import os
import glob
import pandas as pd
from openbabel import pybel
from vina import Vina
from concurrent.futures import ProcessPoolExecutor, as_completed

# Install necessary dependencies
!pip install -q openpyxl openbabel vina pandas

# --- CONFIGURATION ---
SDF_FILE = "sancdb_prepared_3d.sdf"
RECEPTOR_FILE = "1HCK_receptor.pdbqt" if os.path.exists("1HCK_receptor.pdbqt") else "1hck_receptor.pdbqt"
LIGAND_DIR = "sancdb_pdbqt"
OUTPUT_DIR = "docked_poses"
EXCEL_OUTPUT = "1HCK_Detailed_Classical_Hits.xlsx"
CSV_OUTPUT = "1HCK_Detailed_Classical_Hits.csv"

CENTER = [-9.344334, -10.6396675, 8.487167]
GRID_SIZE = [20.0, 20.0, 20.0]

os.makedirs(LIGAND_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. PREPARE LIGANDS: ADD HYDROGENS & STRIP COMPND TAGS
print("Converting SDF library, adding explicit hydrogens, & cleaning headers...")
hydrogen_counts = {}

for i, mol in enumerate(pybel.readfile("sdf", SDF_FILE)):
    ligand_id = f"sanc_{i+1}"
    
    # Add explicit hydrogen atoms for proper charge/docking calculations
    mol.addh()
    
    # Store hydrogen atom count for reporting
    h_count = sum(1 for atom in mol.atoms if atom.atomicnum == 1)
    hydrogen_counts[ligand_id] = h_count
    
    # Write to PDBQT and strip invalid COMPND lines
    pdbqt_str = mol.write("pdbqt")
    clean_lines = [line for line in pdbqt_str.splitlines(keepends=True) if not line.startswith("COMPND")]
    
    out_path = os.path.join(LIGAND_DIR, f"{ligand_id}.pdbqt")
    with open(out_path, "w") as f:
        f.writelines(clean_lines)

print(f"Prepared {len(hydrogen_counts)} sanitized ligands with explicit hydrogens in '{LIGAND_DIR}/'.")

# 2. HEAVY-ATOM RMSD CALCULATION
def calc_heavy_atom_rmsd(ref_path, docked_path):
    try:
        ref_m = next(pybel.readfile("pdbqt", ref_path))
        doc_m = next(pybel.readfile("pdbqt", docked_path))
        ref_coords = [a.coords for a in ref_m.atoms if a.atomicnum > 1]
        doc_coords = [a.coords for a in doc_m.atoms if a.atomicnum > 1]
        
        if not ref_coords or len(ref_coords) != len(doc_coords):
            return 0.0
            
        sq_dist = sum((r[0]-d[0])**2 + (r[1]-d[1])**2 + (r[2]-d[2])**2 for r, d in zip(ref_coords, doc_coords))
        return round((sq_dist / len(ref_coords)) ** 0.5, 3)
    except Exception:
        return 0.0

# 3. WORKER FUNCTION FOR PARALLEL DOCKING
def dock_and_evaluate(ligand_path):
    ligand_id = os.path.splitext(os.path.basename(ligand_path))[0]
    out_pose_path = os.path.join(OUTPUT_DIR, f"{ligand_id}_docked.pdbqt")
    
    try:
        v = Vina(sf_name='vina')
        v.set_receptor(RECEPTOR_FILE)
        v.compute_vina_maps(center=CENTER, box_size=GRID_SIZE)
        v.set_ligand_from_file(ligand_path)
        v.dock(exhaustiveness=8, n_poses=1)
        v.write_poses(out_pose_path, n_poses=1, overwrite=True)
        
        delta_g = round(float(v.energies()[0][0]), 3)
        rmsd = calc_heavy_atom_rmsd(ligand_path, out_pose_path)
        h_atoms = hydrogen_counts.get(ligand_id, 0)
        
        return {
            "Ligand_ID": ligand_id,
            "Delta_G_kcal_mol": delta_g,
            "Pose_RMSD_A": rmsd,
            "Hydrogen_Atoms": h_atoms
        }
    except Exception:
        return None

# 4. MULTI-CORE PARALLEL DOCKING
ligand_files = glob.glob(os.path.join(LIGAND_DIR, "*.pdbqt"))
num_cores = os.cpu_count() or 4
print(f"Launching parallel virtual screening on {num_cores} CPU cores...")

results = []
with ProcessPoolExecutor(max_workers=num_cores) as executor:
    futures = {executor.submit(dock_and_evaluate, f): f for f in ligand_files}
    completed = 0
    for future in as_completed(futures):
        res = future.result()
        if res:
            results.append(res)
        completed += 1
        if completed % 100 == 0 or completed == len(ligand_files):
            print(f"Progress: {completed}/{len(ligand_files)} compounds processed...")

# 5. SORT & EXPORT RESULTS
df = pd.DataFrame(results).sort_values(by="Delta_G_kcal_mol", ascending=True)

# Save both Excel and CSV formats
df.to_csv(CSV_OUTPUT, index=False)
df.to_excel(EXCEL_OUTPUT, index=False)

print(f"\nVirtual screening completed successfully!")
print(f"Results saved to:\n - {EXCEL_OUTPUT}\n - {CSV_OUTPUT}\n")
print("Top 10 Ranked Hits:")
print(df.head(10))

Converting SDF library, adding explicit hydrogens, & cleaning headers...
Prepared 1002 sanitized ligands with explicit hydrogens in 'sancdb_pdbqt/'.
Launching parallel virtual screening on 32 CPU cores...
Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... done.
done.
done.
done.
don

*******Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... *Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... *Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... *Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... *Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... *Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... *Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... *Computing Vina grid ... ******

*Performing docking (random seed: 1695121919) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
****************************
*
mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -8.845          0          0
Computing Vina grid ... done.
done.
done.
done.
done.
done.
done.
done.
done.
done.
done.
done.
done.
done.
Performing docking (random seed: 1837840636) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
*done.


done.
**Computing Vina grid ... done.
done.
done.
Computing Vina grid ... *Computing Vina grid ... Computing Vina grid ... *done.
done.
done.
Computing Vina grid ... *done.
done.
Computing Vina grid ... **Computing Vina grid ... done.
Computing Vina grid ... done.
*Computing Vina grid ... Computing Vina grid ... *Computing Vina grid ... Computing Vina grid ... *Computing Vina grid ... *done.
*Computing Vina grid ... *Computing Vina grid ... *Computing Vina grid ... *Computing Vina grid ... *Computing Vina grid ... Computing Vina grid ... *Computing Vina grid ... Computing Vina grid ... done.
*done.
Computing Vina grid ... Computing Vina grid ... *done.
Computing Vina grid ... **Computing Vina grid ... **Computing Vina grid ... *********Computing Vina grid ... **Computing Vina grid ... *Computing Vina grid ... **********done.
*****done.
Computing Vina grid ... *done.
done.
*
done.
done.
done.
Performing docking (random seed: 1547747391) ... 
0%   10   20   30   40   50   60   70   80   

done.
*Computing Vina grid ... Computing Vina grid ... *done.
done.
*
mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -10.09          0          0
Computing Vina grid ... Computing Vina grid ... *Computing Vina grid ... *done.
done.
done.
done.
done.
*done.
done.
*Computing Vina grid ... Computing Vina grid ... **Computing Vina grid ... done.
Performing docking (random seed: 1804480169) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
*

...

**done.
*done.
**
mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -8.824          0          0
Computing Vina grid ... ********done.
*******Computing Vina grid ... ********Computing Vina grid ... *Computing Vina grid ... *done.
***
done.
done.
**Computing Vina grid ... *done.
****
mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -4.692          0          0
Computing Vina grid ... *done.
***Computing Vina grid ... *Computing Vina grid ... done.
*Computing Vina grid ... **done.
Computing Vina grid ... *****done.
*Computing Vina grid ... done.
done.
**done.
**Computing Vina grid ... ***done.
**Computing Vina grid ... *done.
Computing Vina grid ... **Computing Vina grid ... *Computing Vina grid ... *****Computing Vina grid ... *done.
Computing Vina grid ... Computing Vina grid ... **Computing Vina grid ... *done.
done.
*Pe

*done.
*
**done.
*******Computing Vina grid ... ***done.
*****Computing Vina grid ... **done.
*done.
**
mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -9.471          0          0
Computing Vina grid ... *done.
done.
*********Computing Vina grid ... ****Performing docking (random seed: 313943506) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
******

**Performing docking (random seed: 1346972667) ... 
0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***

Computing Vina grid ... *done.
*Computing Vina grid ... Computing Vina grid ... **
*done.
done.
*done.
Computing Vina grid ... *done.

mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
   1       -8.494          0          0
Computing Vina grid ... *Computing Vina grid ... *Computing Vina grid ... Computing Vina grid ... done.
Computing Vina grid ... done.
done.
done.
done.
done.
*done.
Computing Vina grid ... **Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... Computing Vina grid ... *Computing Vina grid ... *done.
***done.
**Computing Vina grid ... done.
***done.
done.
Computing Vina grid ... *done.
done.
Computing Vina grid ... *Computing Vina grid ... *Computing Vina grid ... done.
Computing Vina grid ... Computing Vina grid ... *done.
*done.
done.
done.
Computing Vina grid ... *Computing Vina grid ... done.
*Computing Vina grid ... done.
Computing Vina grid ... Com

done.
done.
**Computing Vina grid ... done.
Computing Vina grid ... ********done.
****done.
****done.
done.
**done.
done.
*done.
******done.
done.
*done.
***done.
done.
****done.
****done.
***done.
***********************
*****************************************************
******************************************************
***********************************************************************************************************************
**********
*****
*********************************
*************
********************************
*************************
Progress: 1000/1002 compounds processed...
************
***************
Progress: 1002/1002 compounds processed...

Virtual screening completed successfully!
Results saved to:
 - 1HCK_Detailed_Classical_Hits.xlsx
 - 1HCK_Detailed_Classical_Hits.csv

Top 10 Ranked Hits:
   Ligand_ID  Delta_G_kcal_mol  Pose_RMSD_A  Hydrogen_Atoms
24  sanc_118           -10.821       14.674              23
1   sanc_704           -10.085     